# Handling Missing Values

Real-world datasets are rarely complete — sensors fail, people skip survey questions, records get lost in transit. Before any model can be trained, we need to understand *why* data is missing and *how* to deal with it, because the wrong strategy can quietly bias the entire dataset.

This notebook covers:
1. The three mechanisms behind missing data (MCAR, MAR, MNAR)
2. How to detect missing values
3. Deletion-based handling
4. Imputation-based handling (mean, median, mode)
5. Key takeaways + a quick decision guide

## 1. Why Is Data Missing? — The Three Mechanisms

Before choosing *how* to fill or drop missing values, it helps to understand *why* they're missing in the first place. There are three recognized mechanisms:

### 1.1 Missing Completely at Random (MCAR)

The missing value has **nothing to do with any variable** in the dataset — observed or unobserved. It's pure chance.

**Example:** In a health survey, a few participants' answers go missing because of a random data-entry glitch, unrelated to their disease status or any other variable measured.

> If data is truly MCAR, removing the missing rows doesn't introduce bias — you're just losing sample size.

### 1.2 Missing at Random (MAR)

The probability of a value being missing depends on **other observed variables**, but not on the missing value itself.

**Examples:**
- **Income survey:** Participants skip the income question more often based on their *age* or *gender* — not based on their actual income.
- **Medical records:** Younger, healthier patients are less likely to report blood pressure — but the missingness isn't tied to what their blood pressure actually is.

### 1.3 Missing Not at Random (MNAR)

The probability of a value being missing depends on the **value itself** (which we can't observe).

**Example:** In an employee survey, people with *lower* job satisfaction are less likely to report their income. Here the missingness is driven by an unmeasured factor (job satisfaction) that correlates with the value being hidden — this is the trickiest case to handle, since the missing data isn't random at all.

### Quick Comparison

| Mechanism | Depends on | Example | Safe to just drop rows? |
|---|---|---|---|
| **MCAR** | Nothing — pure chance | Sensor glitch, random typo | Yes, generally safe |
| **MAR** | Other observed columns | Age → likelihood of reporting income | Depends — imputation is safer |
| **MNAR** | The missing value itself | High salary → hidden salary | Risky — dropping/imputing naively biases the data |

## 2. Setup

We'll use the built-in **Titanic** dataset from seaborn — it has a nice mix of numerical (`age`) and categorical (`embarked`) columns with real missing values, making it a good playground for this topic.

In [ ]:
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()

## 3. Detecting Missing Values

The first step in handling missing data is always to *see* it clearly.

In [ ]:
# Boolean mask - True wherever a value is missing
df.isnull().head()

In [ ]:
# Count of missing values per column - the actual number we care about
df.isnull().sum()

**Observation:** `age`, `embarked`, `deck`, and `embark_town` all have missing values, with `deck` missing the most by far.

## 4. Deletion-Based Handling

The simplest approach: just drop whatever has missing data. Fast, but it comes at the cost of losing information — use with care, especially if the data isn't MCAR.

In [ ]:
# Original shape before dropping anything
df.shape

In [ ]:
# Row-wise deletion - drops any row that has at least one missing value
df.dropna().shape

In [ ]:
# Column-wise deletion - drops any column that has at least one missing value
df.dropna(axis=1).shape

**Observation:** Column-wise deletion removes entire features (like `age`, which is often a genuinely useful predictor) just because a few values are missing. Generally too blunt for real analysis.

> **Takeaway:** Deletion is easy but wasteful. It's best reserved for cases where missingness is minimal and confirmed (or reasonably assumed) to be MCAR.

## 5. Imputation-Based Handling

Instead of discarding data, we can **fill in** missing values with a reasonable estimate. The right estimate depends on the shape of the data and whether the column is numerical or categorical.

### 5.1 Mean Imputation

Works well when the column is **normally distributed** (no heavy skew, no major outliers) — the mean is a fair representative value in that case.

In [ ]:
# Visualize the distribution of 'age' before deciding on an imputation strategy
sns.histplot(df['age'], kde=True)

In [ ]:
# Fill missing ages with the column mean
df['age_mean'] = df['age'].fillna(df['age'].mean())
df[['age', 'age_mean']].head(10)

**Key insight:** Mean imputation preserves the overall average of the column, but it also artificially shrinks the variance (every missing value collapses to the exact same number), which can understate real-world variability.

### 5.2 Median Imputation

Preferred over the mean when the column has **outliers or skew**, since the median is far less sensitive to extreme values.

In [ ]:
# Fill missing ages with the column median instead
df['age_median'] = df['age'].fillna(df['age'].median())
df[['age', 'age_mean', 'age_median']].head(10)

**Key insight:** Age in the Titanic dataset is *slightly* right-skewed (a long tail of older passengers), so the median tends to sit a little lower than the mean — making it a slightly more robust fill value here.

### 5.3 Mode Imputation

Used for **categorical** columns, where mean/median don't make sense. We simply fill missing values with the most frequently occurring category.

In [ ]:
# Inspect rows where 'embarked' is missing
df[df['embarked'].isnull()]

In [ ]:
# What categories exist in 'embarked'?
df['embarked'].unique()

In [ ]:
# Compute the mode using only the non-missing values
mode_value = df[df['embarked'].notna()]['embarked'].mode()[0]
mode_value

In [ ]:
# Fill missing 'embarked' entries with the mode
df['embarked_mode'] = df['embarked'].fillna(mode_value)
df[['embarked', 'embarked_mode']].head(10)

**Key insight:** Mode imputation is straightforward, but if one category dominates the column, it can make that category look even more common than it really is — worth checking the class balance first.

## 6. Key Takeaways

- **MCAR** — Missingness has nothing to do with any variable in the dataset. Purely random.
- **MAR** — Missingness depends on another *observed* variable, but not on the missing value itself.
- **MNAR** — Missingness depends on the missing value itself (people tend to hide data *because* of what the value is).
- **Deletion** is quick but wastes data — safest only under MCAR and when missingness is minimal.
- **Mean imputation** suits normally distributed numerical data.
- **Median imputation** suits numerical data with outliers/skew.
- **Mode imputation** suits categorical data.

## 7. Decision Guide

```
                    Why is data missing?
                            │
        ┌───────────────────┼───────────────────┐
        │                   │                   │
        ▼                   ▼                   ▼
      MCAR                MAR                 MNAR
   (Random)          (Depends on          (Depends on
                     another column)         itself)

  Coffee spill      Gender → Salary      High salary → hidden
  Sensor failure    Age → Weight         High weight → hidden
  Random typo       Department → Bonus   Low marks → hidden
```

**Rule of thumb:** the more the missingness is tied to the value itself (MNAR), the more careful you need to be — naive deletion or imputation can silently bias your dataset toward the values that *do* get reported.